# 03 — MGSIM bed realizations for lake 1951 (2019cw428)

Conditional stochastic bed topography on the **exact Elmer DEM lattice**
(166×166 @ 150 m), using `multigrid_sgsim` (the MGSIM package) with the
gridded radar picks from notebook 02 as conditioning data.

**The chain, and every choice made in it:**
1. **Trend = the published bed DEM** (the sif's own `.dat`, copied by
   `../scripts/copy_source_dems.sh`). Realizations are then *published bed +
   radar-conditioned stochastic departure* — exactly the "perturb the bed,
   hold everything else" story.
2. **Datum**: the radar bed (`BED_HM`, WGS-84 ellipsoidal) is aligned to the
   published datum by subtracting the *median* radar−trend offset (captures
   the ~28 m geoid undulation plus any systematic BedMachine bias in one
   robust number).
3. **Conditioning thinning**: the gridded cells sit 1 px apart along the
   lines — near-duplicate rows that make Gaussian kriging systems nearly
   singular (the classic instability). A checkerboard decimation to
   ≥ 2 px (300 m along-line) keeps the pinning and fixes the numerics.
4. **Variogram on the residuals** (normal-score transformed, as MGSIM
   simulates them): empirical omnidirectional + along/across-flow
   directional shown; adopted model is **Gaussian with a 1%-of-sill
   nugget** — Gaussian for smooth realizations, the small nugget purely
   as a kriging stabilizer (~8 m node-scale noise, sub-visible against
   the 50–100 m relief; the ~10 m within-cell scatter from notebook 02
   is measurement noise, not simulated texture).
5. **Multigrid schedule**: **[3, 1] px** (450 → 150 m). Pilot runs showed
   the 4-level zero-nugget first cut doubled the node-scale roughness
   and inflated far-from-data variance ~2× (each pass beyond the
   variogram range adds an independent ~sill draw); the 2-level version
   is calibrated (departure std ≈ data residual std) and smoother.
6. **N = 10 seeded realizations** (seeds 0–9, exactly reproducible;
   per-seed cache in `real_cache/` makes re-runs instant).
7. **Edge taper**: the stochastic departure ramps to zero over the outer
   **2 km** of the domain, so every realization equals the published bed at
   the walls → identical Elmer boundary conditions across the ensemble. A
   minimum-ice-thickness guard (20 m) protects the solver.

Coordinates in this notebook are lattice pixels (1 px = 150 m); figure
labels convert to km. Needs `multigrid_sgsim`, gstatsim, skgstat and gstools installed.

In [ ]:
import io, json, time, contextlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DERIVED = Path('../data/derived')
SRC = Path('../data/source_copies')
FIGS = Path('../figures'); FIGS.mkdir(exist_ok=True)

N_GRID, PX_M = 166, 150.0

def read_dat(path):
    xyz = np.loadtxt(path)
    assert xyz.shape == (N_GRID * N_GRID, 3)
    # rows are x-fastest: row = iy*166 + ix
    return xyz[:, 2].reshape(N_GRID, N_GRID)

trend = read_dat(SRC / '2019cw428_bed_d25km_h1004_a0.792.dat')      # [iy, ix]
surface = read_dat(SRC / '2019cw428_surface_d25km_h1004_a0.792.dat')

gz = np.load(DERIVED / 'gridded_bed_cw1951_150m.npz')
meta02 = json.loads(str(gz['meta']))
I0, I1 = meta02['dem_sub_lattice']            # the Elmer 166x166 sub-lattice
radar_full = gz['bed_hm']                     # 334x334, ellipsoidal
radar = radar_full[I0:I1 + 1, I0:I1 + 1]      # -> [iy, ix] on the DEM lattice
n_good = gz['n_good'][I0:I1 + 1, I0:I1 + 1]

# Copy-provenance check: the .dat trend should match the published nc bed
# over the analysis tile (they share one ancestry).
NC = Path('../../cinematic/source_data/pnas_sims/2019cw428_d25km_r2500m_csi6.5e6_tsic_viscous.nc')
if NC.exists():
    import xarray as xr
    ds = xr.open_dataset(NC)
    pub = ds.topo_bed.isel(case=0, level=0)
    # nc grid: 25 m, |l| <= 7375; DEM nodes inside the tile: every 6th nc cell
    ix = np.arange(N_GRID); l = (ix * PX_M) - 12375.0
    keep = np.abs(l) <= 7375
    from scipy.interpolate import RegularGridInterpolator
    itp = RegularGridInterpolator((ds.y.values, ds.x.values), pub.values,
                                  bounds_error=False)
    yy, xx = np.meshgrid(l[keep], l[keep], indexing='ij')
    diff = trend[np.ix_(keep, keep)] - itp(np.stack([yy.ravel(), xx.ravel()], 1)).reshape(yy.shape)
    print(f"trend .dat vs published nc over the tile: median |diff| = "
          f"{np.nanmedian(np.abs(diff)):.2f} m (should be ~0)")

obs_mask = np.isfinite(radar)
print(f"conditioned DEM nodes: {int(obs_mask.sum())} of {N_GRID**2} "
      f"({100 * obs_mask.mean():.1f}%)")

In [ ]:
# Datum alignment + outlier trim, then the trend/residual picture.
resid_raw = radar - trend
datum_shift = float(np.nanmedian(resid_raw))
radar_aligned = radar - datum_shift
resid = radar_aligned - trend
lo, hi = np.nanpercentile(resid, [0.25, 99.75])
n_trim = int(np.sum((resid < lo) | (resid > hi)))
resid_t = np.where((resid >= lo) & (resid <= hi), resid, np.nan)
obs_mask = np.isfinite(resid_t)
print(f"datum shift (median radar − trend): {datum_shift:+.1f} m "
      f"(geoid ≈ +28 m + systematic BedMachine offset)")
print(f"outlier trim at [{lo:.0f}, {hi:.0f}] m: {n_trim} cells dropped "
      f"→ {int(obs_mask.sum())} conditioning nodes")
print(f"residual std: {np.nanstd(resid_t):.1f} m")
# checkerboard-thin the conditioning set to >= 2 px separation along
# lines (Gaussian-kriging stability; see intro)
chk = (np.add.outer(np.arange(N_GRID), np.arange(N_GRID)) % 2) == 0
cond_mask = obs_mask & chk
print(f"conditioning after checkerboard thinning: {int(cond_mask.sum())} nodes")

ext = [0, (N_GRID - 1) * PX_M / 1e3] * 2
fig, axs = plt.subplots(1, 3, figsize=(16.5, 5.2), constrained_layout=True)
b0 = axs[0].imshow(trend, origin='lower', extent=ext, cmap='gist_earth')
axs[0].set_title('trend = published bed DEM (the sif .dat)')
fig.colorbar(b0, ax=axs[0], shrink=0.8, label='m')
vv = np.nanpercentile(np.abs(resid_t), 98)
r0 = axs[1].imshow(resid_t, origin='lower', extent=ext, cmap='RdBu_r', vmin=-vv, vmax=vv)
axs[1].set_title('conditioning residuals (radar − trend, datum-aligned)')
fig.colorbar(r0, ax=axs[1], shrink=0.8, label='m')
axs[2].hist(resid_t[np.isfinite(resid_t)], bins=60, color='#888')
axs[2].axvline(0, color='k', lw=1)
axs[2].set_title(f"residual histogram (std {np.nanstd(resid_t):.0f} m)")
axs[2].set_xlabel('m')
for a in axs[:2]: a.set_xlabel('DEM x [km]')
axs[0].set_ylabel('DEM y [km]')
fig.savefig(FIGS / 'mgsim_trend_residuals.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Variogram of the NST residuals — what MGSIM actually simulates.
from sklearn.preprocessing import QuantileTransformer
from multigrid_sgsim import cluster_variogram

iy, ix = np.where(cond_mask)
obs = pd.DataFrame({'x': ix.astype(float), 'y': iy.astype(float),
                    'residual': resid_t[cond_mask], 'cluster': 0})
nst = QuantileTransformer(output_distribution='normal',
                          n_quantiles=min(1000, len(obs))
                          ).fit(obs[['residual']].values)
obs['Nresidual'] = nst.transform(obs[['residual']].values).ravel()
# variogram fits use a seeded 4000-pt subsample (pairwise cost is O(n^2));
# the NST and the simulation use ALL conditioning points
vsub = obs.sample(n=min(4000, len(obs)), random_state=0)

MAXLAG, N_LAGS = 60.0, 40           # 60 px = 9 km
V_omni = cluster_variogram(vsub, 'Nresidual', 0, MAXLAG, N_LAGS, 'gaussian')
V_along = cluster_variogram(vsub, 'Nresidual', 0, MAXLAG, N_LAGS, 'gaussian',
                            azimuth=0.0, bandwidth=20.0)    # +x = along-flow
V_across = cluster_variogram(vsub, 'Nresidual', 0, MAXLAG, N_LAGS, 'gaussian',
                             azimuth=90.0, bandwidth=20.0)
rng_o, sill_o, nug_o = V_omni.parameters
print(f"fitted omni Gaussian: range {rng_o:.1f} px = {rng_o * PX_M / 1e3:.2f} km, "
      f"sill {sill_o:.3f}, nugget {nug_o:.3f}")
print(f"fitted along-flow range {V_along.parameters[0] * PX_M / 1e3:.2f} km · "
      f"across-flow {V_across.parameters[0] * PX_M / 1e3:.2f} km")

# ADOPTED model — Gaussian for smoothness, isotropic at the omni range,
# 1%-of-sill nugget purely as a kriging stabilizer.
vario = [0.0, 0.01 * float(sill_o), float(rng_o), float(rng_o), float(sill_o), 'Gaussian']
print('adopted vario [az, nug, major, minor, sill, model]:',
      [round(v, 3) if isinstance(v, float) else v for v in vario])

import skgstat as skg
h = np.linspace(0, MAXLAG, 200)
gam = [skg.models.gaussian(hh, rng_o, sill_o, 0.01 * sill_o) for hh in h]
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.scatter(V_omni.bins, V_omni.experimental, c='k', s=32, zorder=3,
           label='empirical (omni)')
ax.scatter(V_along.bins, V_along.experimental, c='#c0392b', s=16, marker='^',
           label='empirical along-flow')
ax.scatter(V_across.bins, V_across.experimental, c='#2980b9', s=16, marker='v',
           label='empirical across-flow')
ax.plot(h, gam, lw=2.5, color='#444', label='adopted: Gaussian, 1% nugget')
sec = ax.secondary_xaxis('top', functions=(lambda p: p * PX_M / 1e3,
                                           lambda k: k * 1e3 / PX_M))
sec.set_xlabel('lag [km]')
ax.set_xlabel('lag [px, 150 m each]'); ax.set_ylabel('γ(h) [NST units]')
ax.set_title('residual variogram — smooth-by-design model choice')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIGS / 'mgsim_variogram.png', dpi=160)
plt.show()

In [ ]:
# The multigrid schedule: conditioning data subsampled coarse→fine.
from multigrid_sgsim.sampling import subsample_dataframe

MG_RESOLS = [3, 1]     # px → 450, 150 m (see intro for the pilot story)
fig, axs = plt.subplots(1, len(MG_RESOLS), figsize=(17, 4.6),
                        constrained_layout=True)
for ax, r in zip(axs, MG_RESOLS):
    sub = obs if r == 1 else subsample_dataframe(
        obs, column_for_sampling='residual', spacing=r,
        random_state=np.random.default_rng(0))
    ax.scatter(obs.x * PX_M / 1e3, obs.y * PX_M / 1e3, s=1, c='0.85')
    ax.scatter(sub.x * PX_M / 1e3, sub.y * PX_M / 1e3, s=2.5, c='#c0392b')
    ax.set_title(f"level {r} px = {r * PX_M:.0f} m · {len(sub)} pts")
    ax.set_aspect('equal'); ax.set_xlabel('DEM x [km]')
axs[0].set_ylabel('DEM y [km]')
fig.suptitle('multigrid conditioning schedule (red = used at that level)')
fig.savefig(FIGS / 'mgsim_schedule.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# N seeded MGSIM realizations (cached per seed — delete real_cache/ to redo).
from multigrid_sgsim import mgsim

N_REAL, K_NEIGHBORS, RADIUS = 10, 12, 50.0   # radius 50 px = 7.5 km
CACHE = DERIVED / 'real_cache'; CACHE.mkdir(exist_ok=True)

gy, gx = np.mgrid[0:N_GRID, 0:N_GRID]
df_grid = pd.DataFrame({
    'x': gx.ravel().astype(float), 'y': gy.ravel().astype(float),
    'trend': trend.ravel(), 'cluster': 0,
    'value': np.where(cond_mask, radar_aligned, np.nan).ravel(),
})
df_grid['set'] = np.isfinite(df_grid['value']).astype(int)
# (outlier-trimmed cells never entered obs_mask, so they are already
#  NaN-valued non-conditioning nodes here)
print(f"{int(df_grid['set'].sum())} conditioning nodes · {len(df_grid)} grid nodes")

reals = np.full((N_REAL, N_GRID, N_GRID), np.nan, dtype=np.float32)
for i in range(N_REAL):
    f = CACHE / f'real_seed{i:02d}.npz'
    if f.exists():
        reals[i] = np.load(f)['bed']
        print(f"seed {i}: cached")
        continue
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):
        out = mgsim(mg_resols=MG_RESOLS, df_xyvtcs=df_grid.copy(), vario=vario,
                    nst_trans=nst, seed=i, xx='x', yy='y', kk='cluster',
                    num_points=K_NEIGHBORS, radius=RADIUS, sgs_or_krige='sgs',
                    clip_nst=True, clip_percentile=98.0)
    reals[i] = out['newtrend'].values.astype(np.float32).reshape(N_GRID, N_GRID)
    np.savez_compressed(f, bed=reals[i])
    print(f"seed {i}: {time.time() - t0:.0f} s")

# conditioning check: SGS honors the data up to the 98% NST clip — the
# extreme-tail observations (line-crossing disagreements) are deliberately
# not reproduced exactly, so report the whole error distribution.
errs = np.abs(reals[:, cond_mask] - radar_aligned[cond_mask][None, :]).ravel()
q = np.percentile(errs, [50, 95, 99])
print(f"|realization − data| at conditioning nodes: median {q[0]:.2f} m, "
      f"p95 {q[1]:.1f} m, p99 {q[2]:.1f} m, max {errs.max():.0f} m "
      f"({100 * float((errs > 10).mean()):.1f}% of node-checks above 10 m)")

In [ ]:
# Edge taper (stochastic departure → 0 at the walls) + thickness guard.
TAPER_M, MIN_ICE_M = 2000.0, 20.0
d_edge = np.minimum.reduce([gx, gy, N_GRID - 1 - gx, N_GRID - 1 - gy]) * PX_M
w = 0.5 - 0.5 * np.cos(np.pi * np.clip(d_edge / TAPER_M, 0, 1))

beds = np.empty_like(reals)
n_clip = 0
for i in range(N_REAL):
    b = trend + w * (reals[i] - trend)
    cap = surface - MIN_ICE_M
    n_clip += int(np.sum(b > cap))
    beds[i] = np.minimum(b, cap)
print(f"taper: {TAPER_M:.0f} m cosine ramp — every bed equals the published "
      f"bed at the walls")
print(f"min-thickness guard ({MIN_ICE_M:.0f} m): {n_clip} node-clips across "
      f"the whole ensemble")
print(f"departure std inside the taper=1 region: "
      f"{float(np.std((beds - trend)[:, w == 1])):.1f} m")

In [ ]:
# The ensemble: departures from the published bed, then mean/spread.
vv = float(np.nanpercentile(np.abs(beds - trend), 99))
fig, axs = plt.subplots(2, 5, figsize=(17, 7.2), constrained_layout=True)
for i, ax in enumerate(axs.ravel()):
    im = ax.imshow(beds[i] - trend, origin='lower', extent=ext,
                   cmap='RdBu_r', vmin=-vv, vmax=vv)
    ax.set_title(f"seed {i}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, shrink=0.7, label='bed − published bed [m]')
fig.suptitle('10 conditioned bed realizations — departure from the published bed')
fig.savefig(FIGS / 'mgsim_gallery.png', dpi=160, bbox_inches='tight')
plt.show()

fig, axs = plt.subplots(1, 3, figsize=(16.5, 5), constrained_layout=True)
m0 = axs[0].imshow(beds.mean(0) - trend, origin='lower', extent=ext,
                   cmap='RdBu_r', vmin=-vv, vmax=vv)
axs[0].set_title('ensemble mean − published bed')
fig.colorbar(m0, ax=axs[0], shrink=0.8, label='m')
s0 = axs[1].imshow(beds.std(0), origin='lower', extent=ext, cmap='magma')
axs[1].set_title('ensemble spread (std)')
fig.colorbar(s0, ax=axs[1], shrink=0.8, label='m')
axs[1].scatter(obs.x * PX_M / 1e3, obs.y * PX_M / 1e3, s=0.4, c='w', alpha=0.35)
b0 = axs[2].imshow(beds[0], origin='lower', extent=ext, cmap='gist_earth')
axs[2].set_title('realization 0, full bed')
fig.colorbar(b0, ax=axs[2], shrink=0.8, label='m')
for a in axs: a.set_xlabel('DEM x [km]')
axs[0].set_ylabel('DEM y [km]')
fig.savefig(FIGS / 'mgsim_ensemble.png', dpi=160, bbox_inches='tight')
plt.show()
print('The spread collapses to ~0 on the radar lines (white dots) and at the\n'
      'tapered walls, and opens up between the lines — as it should.')

In [ ]:
# How uncertainty grows with distance from the radar lines — and an honest
# look at the multigrid variance behavior: each MG pass adds an
# independent ~sill-variance draw at nodes farther than the variogram
# range from any line, so the far-field spread exceeds the 57 m data std.
# For UQ framing that is a conservative (wide) prior between the lines;
# tightening it is a one-knob change (fewer MG levels).
from scipy.spatial import cKDTree
tree = cKDTree(np.column_stack([obs.x, obs.y]) * PX_M)
d_node, _ = tree.query(np.column_stack([gx.ravel(), gy.ravel()]) * PX_M)
d_node = d_node.reshape(N_GRID, N_GRID)
spread = beds.std(0)
core = w == 1
bins = [0, 250, 500, 1000, 1500, 2000, 3000, 4000, 8000]
print(f"{'dist to nearest line':<22}{'nodes':>7}{'spread std [m]':>16}")
mids, vals = [], []
for a, b in zip(bins[:-1], bins[1:]):
    m = core & (d_node >= a) & (d_node < b)
    if m.sum() == 0: continue
    mids.append((a + b) / 2); vals.append(float(spread[m].mean()))
    print(f"{a/1e3:>5.2f}–{b/1e3:<4.2f} km        {int(m.sum()):>7}{vals[-1]:>16.1f}")
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(np.array(mids) / 1e3, vals, 'o-', color='#6b3fa0')
ax.axhline(np.nanstd(resid_t), color='k', ls='--', lw=1,
           label=f'conditioning-data residual std ({np.nanstd(resid_t):.0f} m)')
ax.axvline(rng_o * PX_M / 1e3, color='0.5', ls=':', lw=1.2,
           label=f'variogram range ({rng_o * PX_M / 1e3:.1f} km)')
ax.set_xlabel('distance to nearest radar line [km]')
ax.set_ylabel('ensemble spread [m]')
ax.set_title('bed uncertainty vs distance from data (taper-free interior)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIGS / 'mgsim_spread_vs_distance.png', dpi=160)
plt.show()

In [ ]:
# Along-flow cross-section through the lake (LY = 0 → iy = 82).
IY = 82
xkm = np.arange(N_GRID) * PX_M / 1e3
fig, ax = plt.subplots(figsize=(15, 5.5))
for i in range(N_REAL):
    ax.plot(xkm, beds[i, IY], lw=0.9, alpha=0.75)
ax.plot(xkm, trend[IY], 'k', lw=2.4, label='published bed (trend)')
row_obs = obs[np.abs(obs.y - IY) <= 1]
pick_z = row_obs.residual.values + trend[row_obs.y.values.astype(int),
                                        row_obs.x.values.astype(int)]
ax.scatter(row_obs.x * PX_M / 1e3, pick_z, s=22, c='#c0392b', zorder=5,
           label='radar picks (±1 row)')
ax.plot(xkm, surface[IY], color='0.5', lw=1.4, ls='--', label='surface')
ax.set_xlabel('DEM x — along-flow [km]'); ax.set_ylabel('elevation [m]')
ax.set_title('10 bed realizations along the flowline through the lake center')
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(FIGS / 'mgsim_xsection.png', dpi=160)
plt.show()

In [ ]:
# Package the ensemble + write sif-format .dat DEMs for the Elmer runs.
np.savez_compressed(
    DERIVED / 'bed_realizations_cw1951.npz',
    beds=beds, trend=trend, surface=surface, taper=w.astype(np.float32),
    obs_mask=obs_mask,
    meta=json.dumps({
        'n_real': N_REAL, 'seeds': list(range(N_REAL)),
        'datum_shift_m': datum_shift,
        'vario': [round(float(v), 4) if not isinstance(v, str) else v for v in vario],
        'vario_units': 'px (150 m)', 'mg_resols_px': MG_RESOLS,
        'num_points': K_NEIGHBORS, 'radius_px': RADIUS,
        'nst': 'QuantileTransformer(normal), clip 98%',
        'taper_m': TAPER_M, 'min_ice_m': MIN_ICE_M,
        'trend': '2019cw428_bed_d25km_h1004_a0.792.dat (published)',
        'conditioning': 'notebook-02 gridded radar bed (BED_HM), datum-aligned',
    }))

DEMS = DERIVED / 'dems'; DEMS.mkdir(exist_ok=True)
coords = np.arange(N_GRID) * PX_M
cx, cy = np.meshgrid(coords, coords)   # x-fastest rows, like the source .dat
for i in range(N_REAL):
    out = np.column_stack([cx.ravel(), cy.ravel(), beds[i].ravel()])
    np.savetxt(DEMS / f'2019cw428_bed_d25km_h1004_a0.792_mgsim{i:02d}.dat',
               out, fmt='%f %f %f')
print(f"wrote bed_realizations_cw1951.npz + {N_REAL} sif-format DEMs in {DEMS}")
print('DEM filenames keep the published h/alpha tokens so the sif MATC paths\n'
      'need only the basename swapped.')

## Next — step 4: the Elmer ensemble

For each of the 10 DEMs: copy the published viscous setup (sif + geo + the
other DEMs) into a run workspace, point `bedDEM_path` at the realization,
regenerate the mesh once (shared — the footprint doesn't depend on the bed),
and run serial `ElmerSolver` in the `andrewdnolan/elmerice` container —
everything else (Csi = 6.5e6, SICOPOLIS temperature, BCs) pinned to the
published configuration. Then `make_nc_viscous`-style post-processing per
run → the surface-stress ensemble.